# Monte Carlo Example

In [1]:
import pyblp
import sys
import numpy as np
sys.path.append('/home/md/Dropbox/Projects/pyRVtest')
import pyRVtest
import pandas as pd

pyblp.options.digits = 2
pyblp.options.verbose = False

## DGP

In [2]:
rng = np.random.default_rng(seed=0)

#Market structure
F = 5           # number of firms (constant across markets)
J_min = 20      # minimum products per market
J_max = 40      # maximum products per market
target_obs = 10000
T = target_obs // ((J_min + J_max) // 2)
J_per_market = rng.integers(J_min, J_max + 1, size=T)
market_ids = np.repeat(np.arange(T), J_per_market)
firm_ids = np.concatenate([np.arange(J_t) % F for J_t in J_per_market])
id_data = pd.DataFrame({'market_ids': market_ids, 'firm_ids': firm_ids})
integration = pyblp.Integration('product', 9)
X1=pyblp.Formulation('1 + prices + x1 + x2')
X2=pyblp.Formulation('0 + x1')
X3=pyblp.Formulation('1  + z')

#Simulation
simulation = pyblp.Simulation(
   product_formulations=(X1,X2,X3),
   beta=[0, -2, 1, 1],
   sigma=1,
   xi_variance=0.2,
   omega_variance=0.2,
   correlation=0,
   gamma=[1, 4],
   product_data=id_data,
   integration=integration,
   seed=0
)
simulation_results = simulation.replace_endogenous()
data = pd.DataFrame(pyblp.data_to_dict(simulation_results.product_data))

#Demand instruments
local_instruments = pyblp.build_differentiation_instruments(
    pyblp.Formulation('0+ x1+x2+z'),
    data,version='local'
)
for i, column in enumerate(local_instruments.T):
    data[f'demand_instruments{i}'] = column
data[f'demand_instruments{i+1}']=data['z']

#Test instruments
test_instruments = pyblp.build_blp_instruments(
    pyblp.Formulation('0+ x1+x2'),
    data
)    
for i, column in enumerate(test_instruments.T):
    data[f'test_instruments{i}'] = column
data['clustering_ids']=data['market_ids']

data.head()

,prices,shares,market_ids,firm_ids,x1,x2,z,demand_instruments0,demand_instruments1,demand_instruments2,demand_instruments3,demand_instruments4,demand_instruments5,demand_instruments6
0,4.262680,0.000265,0,0,0.669116,0.506390,0.705213,5.0,4.0,7.0,25.0,26.0,20.0,0.705213
1,1.951446,0.062931,0,1,0.955594,0.641874,0.017901,2.0,3.0,3.0,16.0,18.0,12.0,0.017901
2,4.469557,0.000093,0,2,0.696314,0.044243,0.629339,5.0,4.0,5.0,24.0,19.0,26.0,0.629339
3,2.622310,0.004278,0,3,0.242824,0.333540,0.304242,2.0,6.0,4.0,19.0,27.0,19.0,0.304242
4,4.605049,0.000149,0,4,0.081143,0.319529,0.894633,1.0,6.0,3.0,13.0,27.0,17.0,0.894633


## Demand estimation

In [3]:
problem = pyblp.Problem(
       (X1,X2),
    product_data=data , integration=integration
)
pyblp_results = problem.solve(
    sigma=0.5, method='1s'
)
print(pyblp_results)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 1    +3.8E-01     +1.2E-09     +8.3E+01     0         +2.9E+04          +4.6E+03     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:08       Yes          5             9          13794        44102   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      x1    
------  ----------
  x1     +1.0E+00 
        (+6.9E-02)

Beta Estimates (Robust SEs in Parentheses):
    1         prices        x1          x2    
----------  ----------  ----------  ----------
 -1.4E-02    -2.0E+00 

## Conduct Test

In [4]:
testing_problem = pyRVtest.Problem(
    cost_formulation = (
        pyRVtest.Formulation('1+z' )
    ),
    instrument_formulation = (
        pyRVtest.Formulation('0+test_instruments0+test_instruments1+test_instruments2+test_instruments3')
    ),
    model_formulations = (
        pyRVtest.ModelFormulation(model_downstream='bertrand', ownership_downstream='firm_ids'),
        pyRVtest.ModelFormulation(model_downstream='monopoly', ownership_downstream='firm_ids')
    ),
    product_data = data,
    demand_results = pyblp_results
)

testing_results = testing_problem.solve(
    demand_adjustment=True,
    clustering_adjustment=True
)
print(testing_results)

Computing Markups ... 
Total Time is ... 1.9138538837432861


Testing Results - Instruments z0:
  TRV:                 |   F-stats:                 |    MCS:                
--------  ---  ------  |  ----------  ---  -------  |  --------  ------------
 models    0     1     |    models     0      1     |   models   MCS p-values
--------  ---  ------  |  ----------  ---  -------  |  --------  ------------
   0      nan  -3.557  |      0       nan   13.2    |     0          1.0     
                       |                   *** ^^^  |                        
   1      nan   nan    |      1       nan    nan    |     1          0.0     
                       |                            |                        
Significance of size and power diagnostic reported below each F-stat                                                                   
*, **, or *** indicate that F > cv for a worst-case size of 0.125, 0.10, and 0.075 given d_z and rho                                            